In [ ]:
import os
import random
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from mtcnn import MTCNN
from tensorflow.keras import layers, callbacks, mixed_precision
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve
)

In [ ]:
CELEB_REAL_DIR  = "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real"
CELEB_FAKE_DIR  = "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis"
FF_BASE         = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
FF_REAL_DIR     = f"{FF_BASE}/original"
FF_DEEPFAKES_DIR = f"{FF_BASE}/Deepfakes"
FF_FACESWAP_DIR = f"{FF_BASE}/FaceSwap"

REAL_SOURCES = [FF_REAL_DIR, CELEB_REAL_DIR]
FAKE_SOURCES = [FF_DEEPFAKES_DIR, FF_FACESWAP_DIR, CELEB_FAKE_DIR]

BASE_OUTPUT = "/kaggle/working/dataset"
TRAIN_REAL  = f"{BASE_OUTPUT}/train/real"
TRAIN_FAKE  = f"{BASE_OUTPUT}/train/fake"
VAL_REAL    = f"{BASE_OUTPUT}/val/real"
VAL_FAKE    = f"{BASE_OUTPUT}/val/fake"
TEST_REAL   = f"{BASE_OUTPUT}/test/real"
TEST_FAKE   = f"{BASE_OUTPUT}/test/fake"

for folder in [TRAIN_REAL, TRAIN_FAKE, VAL_REAL, VAL_FAKE, TEST_REAL, TEST_FAKE]:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully")

In [ ]:
IMG_SIZE = (299, 299)

detector = MTCNN()

def extract_face(frame, target_size=299):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(rgb)
    if not results:
        return None
    x, y, w, h = max(results, key=lambda r: r['box'][2] * r['box'][3])['box']
    x, y = max(0, x), max(0, y)
    ih, iw = frame.shape[:2]
    w, h = min(iw - x, w), min(ih - y, h)
    face = frame[y:y+h, x:x+w]
    if face.shape[0] < 50 or face.shape[1] < 50:
        return None
    return cv2.resize(face, (target_size, target_size))

In [ ]:
def get_all_video_paths(root_dir, max_videos=500):
    paths = []
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.endswith((".mp4", ".avi", ".mov", ".mkv")):
                paths.append(os.path.join(root, f))
    return paths[:max_videos]

def split_videos(paths):
    train, tmp = train_test_split(paths, test_size=0.2, random_state=42)
    val, test  = train_test_split(tmp,   test_size=0.5, random_state=42)
    return train, val, test

def process_single_video(video_path, output_dir, label, max_frames=12):
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    frame_interval = max(1, fps)   # 1 frame per second
    frame_count, saved = 0, 0
    ok, frame = cap.read()
    while ok and saved < max_frames:
        if frame_count % frame_interval == 0:
            face = extract_face(frame)
            if face is not None:
                name = f"{label}_{os.path.basename(video_path)}_{saved}.jpg"
                cv2.imwrite(os.path.join(output_dir, name), face)
                saved += 1
        ok, frame = cap.read()
        frame_count += 1
    cap.release()
    return saved


def extract_frames_from_video_list(video_list, output_dir, label, max_frames=12):
    print(f"Processing {len(video_list)} videos → {output_dir}")
    total = 0
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
        futures = [ex.submit(process_single_video, vp, output_dir, label, max_frames)
                   for vp in video_list]
        for f in tqdm(futures, desc=f"Extracting {label}"):
            total += f.result()
    print(f"Saved {total} frames → {output_dir}")

def already_extracted():
    return (len(os.listdir(TRAIN_REAL)) > 100 and
            len(os.listdir(TRAIN_FAKE)) > 100)

if not already_extracted():
    for sources, label_prefix, dirs in [
        (REAL_SOURCES, "real", (TRAIN_REAL, VAL_REAL, TEST_REAL)),
        (FAKE_SOURCES, "fake", (TRAIN_FAKE, VAL_FAKE, TEST_FAKE)),
    ]:
        all_videos = []
        for src in sources:
            all_videos.extend(get_all_video_paths(src, max_videos=400))
        tr, va, te = split_videos(all_videos)
        for video_list, out_dir in zip([tr, va, te], dirs):
            extract_frames_from_video_list(video_list, out_dir, label_prefix)
else:
    print("Frames already extracted — skipping")

for split in ["train", "val", "test"]:
    for cls in ["real", "fake"]:
        d = f"{BASE_OUTPUT}/{split}/{cls}"
        print(f"{split.upper()} {cls.upper()}: {len(os.listdir(d))}")

In [ ]:
BATCH_SIZE = 16

class JPEGCompressionNoise(layers.Layer):
    def __init__(self, quality_levels=(55, 65, 75, 85), **kwargs):
        super().__init__(**kwargs)
        self.quality_levels = list(quality_levels)

    def _encode_decode(self, image_uint8, quality):
        encoded = tf.image.encode_jpeg(image_uint8, quality=quality)
        return tf.cast(tf.image.decode_jpeg(encoded, channels=3), tf.float32)

    def _apply_to_image(self, image):
        image_uint8 = tf.cast(tf.clip_by_value(image, 0, 255), tf.uint8)
        n = len(self.quality_levels)
        branch_idx = tf.random.uniform([], 0, n, dtype=tf.int32)
        return tf.switch_case(
            branch_idx,
            branch_fns=[
                lambda q=q: self._encode_decode(image_uint8, q)
                for q in self.quality_levels
            ]
        )

    def call(self, x, training=False):
        if not training:
            return x
        return tf.map_fn(self._apply_to_image, x, dtype=tf.float32)

    def get_config(self):
        return {**super().get_config(), "quality_levels": self.quality_levels}


data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15),
    JPEGCompressionNoise(quality_levels=(55, 65, 75, 85)),
    layers.GaussianNoise(0.03),
])

def xception_preprocess(x):
    """Scale [0, 255] → [-1, 1] as Xception expects."""
    return tf.keras.applications.xception.preprocess_input(x)

def prepare(ds, augment=False):
    # DO NOT cache — 15K × 299×299×3 float32 = ~20GB, exceeds Kaggle RAM
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (xception_preprocess(tf.cast(x, tf.float32)), y),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE_OUTPUT}/train", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="int",
    shuffle=True
).unbatch().batch(BATCH_SIZE, drop_remainder=True)   # ← applied here

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE_OUTPUT}/val", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="int"
).unbatch().batch(BATCH_SIZE, drop_remainder=True)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE_OUTPUT}/test", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode="int"
).unbatch().batch(BATCH_SIZE, drop_remainder=False)  # keep all test samples

print("Class names:", train_ds.element_spec)

train_ds = prepare(train_ds, augment=True)
val_ds   = prepare(val_ds)
test_ds  = prepare(test_ds)

In [ ]:
class FFTBranch(layers.Layer):
    def __init__(self, fft_size=75, **kwargs):
        super().__init__(**kwargs)
        self.fft_size = fft_size
        self.conv1 = layers.Conv2D(32, 5, strides=2, activation="relu", padding="same")
        self.bn1   = layers.BatchNormalization()
        self.conv2 = layers.Conv2D(64, 3, strides=2, activation="relu", padding="same")
        self.bn2   = layers.BatchNormalization()
        self.conv3 = layers.Conv2D(128, 3, strides=2, activation="relu", padding="same")
        self.gap   = layers.GlobalAveragePooling2D()
        self.dense = layers.Dense(256, activation="relu")

    def call(self, x, training=False):
        # Downsample to fft_size×fft_size BEFORE FFT — massive memory reduction
        x_small = tf.image.resize(x, [self.fft_size, self.fft_size])

        gray    = tf.image.rgb_to_grayscale(x_small)
        gray    = tf.squeeze(gray, axis=-1)
        gray_c  = tf.cast(gray, tf.complex64)
        fft     = tf.signal.fft2d(gray_c)
        shifted = tf.signal.fftshift(fft)
        mag     = tf.math.log1p(tf.abs(shifted))
        mag     = tf.expand_dims(mag, axis=-1)
        mn      = tf.reduce_min(mag,  axis=[1,2,3], keepdims=True)
        mx      = tf.reduce_max(mag,  axis=[1,2,3], keepdims=True)
        mag     = (mag - mn) / (mx - mn + 1e-8)

        x = self.conv1(mag, training=training)
        x = self.bn1(x,     training=training)
        x = self.conv2(x,   training=training)
        x = self.bn2(x,     training=training)
        x = self.conv3(x,   training=training)
        x = self.gap(x)
        return self.dense(x)

    def get_config(self):
        return {**super().get_config(), "fft_size": self.fft_size}

class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.75, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        p_t    = tf.where(y_true == 1, y_pred, 1.0 - y_pred)
        alpha_t = tf.where(y_true == 0,
                           tf.ones_like(y_true) * self.alpha,
                           tf.ones_like(y_true) * (1.0 - self.alpha))
        loss = -alpha_t * tf.pow(1.0 - p_t, self.gamma) * tf.math.log(p_t)
        return tf.reduce_mean(loss)

    def get_config(self):
        return {**super().get_config(), "gamma": self.gamma, "alpha": self.alpha}

class AccumulatingModel(tf.keras.Model):
    def __init__(self, inner_model, accum_steps=2):
        super().__init__()
        self.inner       = inner_model
        self.accum_steps = accum_steps

    def call(self, x, training=False):
        return self.inner(x, training=training)

    def train_step(self, data):
        x, y = data
        micro_xs = tf.split(x, self.accum_steps, axis=0)
        micro_ys = tf.split(y, self.accum_steps, axis=0)

        accum_vars = [tf.zeros_like(v)
                      for v in self.inner.trainable_variables]
        loss = tf.constant(0.0)

        for mx, my in zip(micro_xs, micro_ys):
            with tf.GradientTape() as tape:
                preds = self.inner(mx, training=True)
                loss  = self.compiled_loss(my, preds)
            grads = tape.gradient(loss, self.inner.trainable_variables)
            accum_vars = [
                av + tf.cast(g, av.dtype) / self.accum_steps
                for av, g in zip(accum_vars, grads)
                if g is not None
            ]

        self.optimizer.apply_gradients(
            zip(accum_vars, self.inner.trainable_variables))

        # ← use self.metrics, not self.compiled_metrics.metrics
        preds_full = self.inner(x, training=False)
        for metric in self.metrics:
            if metric.name == "loss":
                continue
            metric.update_state(y, preds_full)

        return {"loss": loss, **{m.name: m.result() for m in self.metrics if m.name != "loss"}}

    def test_step(self, data):
        x, y  = data
        preds = self.inner(x, training=False)
        loss  = self.compiled_loss(y, preds)

        for metric in self.metrics:
            if metric.name == "loss":
                continue
            metric.update_state(y, preds)

        return {"loss": loss, **{m.name: m.result() for m in self.metrics if m.name != "loss"}}

In [ ]:
base_model = tf.keras.applications.Xception(
    include_top=False,
    weights="imagenet",
    input_shape=(299, 299, 3),
    pooling="avg"
)
base_model.trainable = False
# No tf.recompute_grad — it wraps the model in a raw TF function which
# can't accept KerasTensors during functional graph construction.

inputs           = tf.keras.Input(shape=(299, 299, 3))
spatial_features = base_model(inputs, training=False)
freq_features    = FFTBranch(name="fft_branch", fft_size=75)(inputs)

x = layers.Concatenate()([spatial_features, freq_features])
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)

inner_model = tf.keras.Model(inputs, outputs, name="XceptionFFT_DeepfakeDetector")
inner_model.summary()

# Wrap with gradient accumulation (effective batch = 16 × 2 = 32)
model = AccumulatingModel(inner_model, accum_steps=2)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4,
        epsilon=1e-4,
        clipnorm=1.0
    ),
    loss=BinaryFocalLoss(gamma=2.0, alpha=0.75),
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision"),
    ]
)
print("Model built successfully.")

In [ ]:
class InnerModelCheckpoint(tf.keras.callbacks.Callback):
    """Saves inner_model weights when val_auc improves."""
    def __init__(self, inner, path):
        super().__init__()
        self.inner    = inner
        self.path     = path
        self.best_auc = 0.0

    def on_epoch_end(self, epoch, logs=None):
        auc = logs.get("val_auc", 0.0)
        if auc > self.best_auc:
            self.best_auc = auc
            self.inner.save_weights(self.path)
            print(f"\nCheckpoint saved (val_auc: {auc:.4f}) → {self.path}")

callbacks_p1 = [
    callbacks.ReduceLROnPlateau(
        monitor="val_auc", factor=0.5, patience=2,
        min_lr=1e-6, verbose=1, mode="max"),
    callbacks.EarlyStopping(
        monitor="val_auc", patience=5,
        restore_best_weights=True, verbose=1, mode="max"),
    InnerModelCheckpoint(inner_model, "./video_model_best.weights.keras"),
]

print("Starting Phase 1 (frozen backbone)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks_p1,
)

print("Phase 1 completed")

In [ ]:
print("Starting Phase 2 (fine-tune last 2 Xception blocks)...")

callbacks_p2 = [
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2,
        min_lr=1e-7, verbose=1, mode="min"),   # mode="min" for loss
    callbacks.EarlyStopping(
        monitor="val_loss", patience=5,
        restore_best_weights=True, verbose=1, mode="min"),
    InnerModelCheckpoint(inner_model, "./video_model_best.weights.h5"),
]

model = AccumulatingModel(inner_model, accum_steps=2)

base_model = inner_model.get_layer("xception")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f"Trainable layers: {sum(1 for l in inner_model.layers if l.trainable)}")

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=5e-5,
        weight_decay=1e-5,
        epsilon=1e-4,
        clipnorm=1.0
    ),
    loss=BinaryFocalLoss(gamma=2.0, alpha=0.75),
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision"),
    ]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks_p2,
)

print("Phase 2 completed")

In [ ]:
print("Tuning classification threshold on validation set...")

val_labels, val_probs = [], []
inner_model.load_weights("./video_model_best.weights.keras")

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    val_probs.extend(preds.flatten())       # ← .flatten() instead of [:, 0]
    val_labels.extend(labels.numpy())

val_labels = np.array(val_labels)
val_probs  = np.array(val_probs)           # P(real) — low value = fake

# Fake probability = 1 - P(real)
fake_probs = 1.0 - val_probs

fpr, tpr, thresholds = roc_curve(val_labels == 0, fake_probs)
auc_score = roc_auc_score(val_labels == 0, fake_probs)
print(f"Validation AUC: {auc_score:.4f}")

# Find threshold on fake_prob that gives recall >= 0.90
best_thresh, best_prec = 0.5, 0.0
for thresh in np.linspace(0.1, 0.9, 81):
    preds_t     = (fake_probs >= thresh).astype(int)
    tp          = np.sum((preds_t == 1) & (val_labels == 0))
    fp          = np.sum((preds_t == 1) & (val_labels == 1))
    fn          = np.sum((preds_t == 0) & (val_labels == 0))
    recall_t    = tp / (tp + fn + 1e-9)
    precision_t = tp / (tp + fp + 1e-9)
    if recall_t >= 0.90 and precision_t > best_prec:
        best_thresh = thresh
        best_prec   = precision_t

print(f"Selected threshold: {best_thresh:.2f}  (Recall ≥ 0.90, Precision ≈ {best_prec:.2f})")
OPTIMAL_THRESHOLD = best_thresh

In [ ]:
print("\nEvaluating on test set...")
all_labels, all_probs = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    all_probs.extend(preds.flatten())       # ← .flatten()
    all_labels.extend(labels.numpy())

all_labels  = np.array(all_labels)
all_probs   = np.array(all_probs)
fake_probs  = 1.0 - all_probs
all_preds   = (fake_probs >= OPTIMAL_THRESHOLD).astype(int)  # 1 = fake

print(classification_report(
    all_labels == 0,   # True if fake
    all_preds,
    target_names=["real", "fake"],
    zero_division=0
))
test_auc = roc_auc_score(all_labels == 0, fake_probs)
print(f"Test AUC:  {test_auc:.4f}")
print(f"Threshold: {OPTIMAL_THRESHOLD:.2f}")

# Confusion matrix
cm = confusion_matrix(all_labels == 0, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Real", "Pred Fake"],
            yticklabels=["Actual Real", "Actual Fake"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
plt.scatter([1-best_prec], [0.90], color="red", zorder=5,
            label=f"Threshold {OPTIMAL_THRESHOLD:.2f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve — Fake Detection")
plt.legend()
plt.show()

In [ ]:
def predict_video(video_path, threshold=OPTIMAL_THRESHOLD):
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    frame_interval = max(1, fps // 2)
    frame_count, faces_batch = 0, []

    ok, frame = cap.read()
    while ok:
        if frame_count % frame_interval == 0:
            face = extract_face(frame)
            if face is not None:
                face = tf.keras.applications.xception.preprocess_input(
                    face.astype(np.float32))
                faces_batch.append(face)
        ok, frame = cap.read()
        frame_count += 1
    cap.release()

    if not faces_batch:
        return "Error: No faces detected."

    batch         = np.array(faces_batch)
    preds         = model.predict(batch, verbose=0).flatten()
    avg_fake_prob = float(np.mean(1.0 - preds))   # 1 - P(real) = P(fake)

    label = "FAKE" if avg_fake_prob >= threshold else "REAL"
    return (f"{'🚨' if label == 'FAKE' else '✅'} {label}  "
            f"(fake prob: {avg_fake_prob:.2%}, threshold: {threshold:.2f})")